In [1]:
import pandas as pd
import torch
import numpy as np
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    Trainer, 
    TrainingArguments,
    DataCollatorWithPadding
)
from sklearn.metrics import f1_score, accuracy_score
import os
import shutil
import gc

# 1. Configuration
class Config:
    MODELS = [
        "xlm-roberta-large", 
        "bert-base-multilingual-cased"
    ]
    MAX_LEN = 512
    BATCH_SIZE = 4
    GRAD_ACCUM = 8
    EPOCHS = 10
    LR = 1e-5
    OUTPUT_DIR = "/kaggle/working/"
    TRAIN_FILE = "/kaggle/input/telugu-xlm/results/train.csv"
    DEV_FILE = "/kaggle/input/telugu-xlm/results/dev.csv"
    TEST_FILE = "/kaggle/input/telugu-xlm/results/test.csv"
    
    LABEL_LIST = [
        'Formal', 'Informal', 'Optimistic', 'Pessimistic', 'Humorous', 
        'Serious', 'Inspiring', 'Authoritative', 'Persuasive'
    ]
    LABEL2ID = {label: i for i, label in enumerate(LABEL_LIST)}
    ID2LABEL = {i: label for i, label in enumerate(LABEL_LIST)}

# 2. Data Loading
def load_data(file_path, is_test=False):
    for enc in ['utf-8-sig', 'utf-8', 'latin-1', 'utf-16']:
        try:
            df = pd.read_csv(file_path, encoding=enc)
            print(f"Successfully loaded {file_path} with {enc} encoding")
            df = df.dropna(subset=['ORIGINAL TRANSCRIPTS', 'CHANGE STYLE'])
            if 'STYLE' in df.columns and not is_test:
                df = df.dropna(subset=['STYLE'])
                df['label'] = df['STYLE'].map(Config.LABEL2ID)
                df = df.dropna(subset=['label'])
            # Put CHANGE STYLE first to prevent truncation of target features
            df['text'] = df['CHANGE STYLE'].fillna('') + " [SEP] " + df['ORIGINAL TRANSCRIPTS'].fillna('')
            return df
        except Exception as e:
            continue
    raise Exception(f"Could not load {file_path}")

# 3. Dataset Component
class TeluguStyleDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=512, text_col='text'):
        self.texts = df[text_col].tolist()
        self.labels = df['label'].tolist() if 'label' in df.columns else None
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        item = {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
        }
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

# 4. Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='macro')
    return {"accuracy": acc, "f1": f1}

def train_and_predict(model_name, train_df, val_df, test_df, device):
    print(f"\n--- Training and predicting with {model_name} ---")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    # Custom format for this specific model's tokenizer
    def format_text(row):
        return str(row['CHANGE STYLE']) + f" {tokenizer.sep_token} " + str(row['ORIGINAL TRANSCRIPTS'])
    
    train_df['formatted_text'] = train_df.apply(format_text, axis=1)
    val_df['formatted_text'] = val_df.apply(format_text, axis=1)
    test_df['formatted_text'] = test_df.apply(format_text, axis=1)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, 
        num_labels=len(Config.LABEL_LIST),
        id2label=Config.ID2LABEL,
        label2id=Config.LABEL2ID
    ).to(device)

    train_dataset = TeluguStyleDataset(train_df, tokenizer, Config.MAX_LEN, text_col='formatted_text')
    val_dataset = TeluguStyleDataset(val_df, tokenizer, Config.MAX_LEN, text_col='formatted_text')
    test_dataset = TeluguStyleDataset(test_df, tokenizer, Config.MAX_LEN, text_col='formatted_text')

    training_args = TrainingArguments(
        output_dir=f"./results_{model_name.replace('/', '_')}",
        num_train_epochs=Config.EPOCHS,
        per_device_train_batch_size=Config.BATCH_SIZE,
        per_device_eval_batch_size=Config.BATCH_SIZE,
        gradient_accumulation_steps=Config.GRAD_ACCUM,
        learning_rate=Config.LR,
        weight_decay=0.01,
        warmup_ratio=0.1,
        logging_steps=50,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        save_total_limit=1,
        fp16=torch.cuda.is_available(),
        report_to="none",
        label_smoothing_factor=0.1,
        save_only_model=True,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    
    # Prediction (return logits for ensembling)
    print(f"Generating test logits for {model_name}...")
    model.eval()
    all_logits = []
    with torch.no_grad():
        for i in range(0, len(test_df), Config.BATCH_SIZE * 2): # Adjusted batch for inference
            batch_df = test_df.iloc[i : i + (Config.BATCH_SIZE * 2)]
            inputs = tokenizer(
                batch_df['formatted_text'].tolist(),
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=Config.MAX_LEN
            ).to(device)
            outputs = model(**inputs)
            all_logits.append(outputs.logits.cpu().numpy())
    
    res = np.concatenate(all_logits, axis=0)
    
    # Cleanup to save space
    del model
    del trainer
    torch.cuda.empty_cache()
    gc.collect()
    
    # Remove results directory if it exists to save space
    if os.path.exists(training_args.output_dir):
        print(f"Cleaning up {training_args.output_dir}...")
        shutil.rmtree(training_args.output_dir)
        
    return res

def main():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    # Load Data
    print("Loading data...")
    train_df = load_data(Config.TRAIN_FILE)
    val_df = load_data(Config.DEV_FILE)
    test_df = load_data(Config.TEST_FILE, is_test=True)

    # Train all models and collect logits
    ensemble_logits = []
    for model_name in Config.MODELS:
        logits = train_and_predict(model_name, train_df, val_df, test_df, device)
        ensemble_logits.append(logits)
    
    # 5. Ensemble: Simple Averaging of Logits
    print("\n--- Performing Ensemble ---")
    avg_logits = np.mean(ensemble_logits, axis=0)
    final_preds = avg_logits.argmax(-1)
    
    # 6. Final Submission CSV
    test_df['STYLE'] = [Config.ID2LABEL[p] for p in final_preds]
    submission_df = test_df[['ID', 'STYLE']]
    submission_df.to_csv("submission.csv", index=False, encoding='utf-8-sig')
    print("Inference complete. Ensemble results saved to submission.csv")

if __name__ == "__main__":
    main()


2026-02-08 15:33:31.163159: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770564811.354630      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770564811.412480      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770564811.908772      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770564811.908812      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770564811.908818      24 computation_placer.cc:177] computation placer alr

Using device: cuda
Loading data...
Successfully loaded /kaggle/input/telugu-xlm/results/train.csv with utf-8-sig encoding
Successfully loaded /kaggle/input/telugu-xlm/results/dev.csv with utf-8-sig encoding
Successfully loaded /kaggle/input/telugu-xlm/results/test.csv with utf-8-sig encoding

--- Training and predicting with xlm-roberta-large ---


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,2.201493,0.099656,0.042982
2,2.197900,2.167506,0.147766,0.069694
3,2.177900,2.106793,0.189003,0.130351
4,2.116100,2.078581,0.209622,0.188058
5,2.046400,2.055540,0.250859,0.239832
6,2.000200,2.013217,0.268041,0.252797
7,1.943000,2.009196,0.261168,0.241700
8,1.912100,1.990636,0.281787,0.274266
9,1.864500,1.980178,0.295533,0.286305
10,1.855800,1.985001,0.271478,0.268095


Generating test logits for bert-base-multilingual-cased...
Cleaning up ./results_bert-base-multilingual-cased...

--- Performing Ensemble ---
Inference complete. Ensemble results saved to submission.csv
